In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType


In [ ]:
def create_spark_session():
    try:
        spark = SparkSession.builder \
            .appName("KafkaConsumer") \
            .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,"
                                            "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0") \
            .config("spark.cassandra.connection.host", "cassandra") \
            .getOrCreate()

        spark.sparkContext.setLogLevel("ERROR")
        print("Spark Session created successfully.")
        return spark
    except Exception as e:
        print(f"Error creating Spark Session: {e}")
        return None

Connect to kafka using a consumer , in this case using spark
<p>This is will be or listener waiting or reading data messages from broker</p>

With `spark.readStream`, Spark continuously monitors the source and processes new data as it arrives

In [ ]:
def connect_to_kafka(spark):
    #Read the micro-batches from kafka broker 
    try:
        spark_df = spark.readStream \
        .format('kafka') \
        .option('kafka.bootstrap.servers', 'kafka:2902') \
        .option('subscribe', 'test-topic') \ 
        .option('startingOffsets', 'earliest') \
        .load()
        print("Connected to Kafka broker successfully.")
        return spark_df

    except Exception as e:
        print(f'Error connecting to kafka broker: {e}')
        return None

Create schema to match incoming data format from airflow dag

structType - defines our columns
<br>structField - a column

In [ ]:
def create_selection_df_from_kafka(spark_df):
    user_schema = StructType([
            StructField("first_name", StringType(), True),
            StructField("last_name", StringType(), True),
            StructField("address", StringType(), True)
        ])

    # Deserialize the binary kafka into a structured dataframe

    try:
        selection_df = spark_df.selectExpr("CAST(value AS STRING)") \
        .select(from_json(col("value"), user_schema).alias("data")) \
        .select("data.*")
        print("Selection DataFrame created successfully.")
        return selection_df
    except Exception as e:
        print(f"Error creating selection DataFrame: {e}")
        return None 

In [ ]:
spark_conn = create_spark_connection()

    if spark_conn is not None:
        # Connect to kafka with spark connection
        spark_df = connect_to_kafka(spark_conn)
        selection_df = create_selection_df_from_kafka(spark_df)
        
        print("Streaming is active. Writing to Cassandra...")
        
        # 5. Write the stream directly into Cassandra
        streaming_query = selection_df.writeStream \
            .format("org.apache.spark.sql.cassandra") \
            .option("checkpointLocation", "/tmp/checkpoint") \
            .option("keyspace", "spark_streams") \
            .option("table", "created_users") \
            .start()

        # Keep the process running waiting for incoming data
        streaming_query.awaitTermination()